<a href="https://colab.research.google.com/github/roser740/Data-Analytics/blob/main/TASQUES%20S13/ETL_meteo.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **ANÀLISI DE LA QUALITAT DE L'AIRE A LA PLANA DE VIC**

# **💢 ETL (EXTRACT, TRANSFORM, LOAD) DADES METEOROLOGIQUES**

# **📚 Llibreries**

In [32]:

import pandas as pd # llibreria per gestionar dataframes
import numpy as np # llibreria per computació numèrica i científica
import codecs # mòdul per codificar i descodificar dades en diferents formats
import seaborn as sns # llibreria de visualització de dades
import matplotlib.pyplot as plt # llibreria per fer gràfiques

from matplotlib.colors import BoundaryNorm, LinearSegmentedColormap

# **⭕ Funcions generals**

In [33]:
def mostrar_dataframe(df, títol):
#Presenta les dades en format tabular amb el títol indicat

    # Mostrem títol
    print("\n" + "="*100)
    print(títol)
    print("="*100)
    # La funció display mostra les dades del dataframe en format taula.
    # Utilitzem la propietat style per amagar la columna de l'índex.
    display(df)

##🟢**Extract: obtenir dades brutes (raw data)**

In [50]:
url = "https://github.com/roser740/Data-Analytics/raw/main/TASQUES%20S13/DADES/"

df_estacions = pd.read_csv(url+"Estacions_meteo.csv", sep = ",", header=0, encoding='latin1')
df_estacions.head(5)

,CODI_ESTACIO,NOM_ESTACIO,CODI_TIPUS,LATITUD,LONGITUD,GeoreferÃ¨ncia,EMPLACAMENT,CODI_MUNICIPI,ALTITUD,NOM_MUNICIPI,CODI_COMARCA,NOM_COMARCA,CODI_PROVINCIA,NOM_PROVINCIA,CODI_XARXA,NOM_XARXA,CODI_ESTAT,NOM_ESTAT,DATA_ALTA,DATA_BAIXA
0,X2,Barcelona - Zoo,A,"41,38943","2,18847",POINT (2.18847 41.38943),Zoo,80193,7,Barcelona,13,BarcelonÃ¨s,8,Barcelona,1,XEMA,1,Desmantellada,22/09/2006,03/10/2024
1,UH,el Montmell,A,"41,34172","1,48769",POINT (1.48769 41.34172),Finca Vallflor - Aigua Viva,430901,547,El Montmell,12,Baix PenedÃ¨s,43,Tarragona,1,XEMA,2,Operativa,13/07/1995,NaN
2,VI,Lleida,A,"41,62918","0,59622",POINT (0.59622 41.62918),Av. Alcalde Rovira Roure,251207,190,Lleida,33,SegriÃ,25,Lleida,1,XEMA,1,Desmantellada,30/03/1990,15/06/2000
3,WT,Malgrat de Mar,A,"41,64706","2,75659",POINT (2.75659 41.64706),CamÃ­ de la Pomereda,81108,3,Malgrat de Mar,21,Maresme,8,Barcelona,1,XEMA,2,Operativa,04/05/2005,NaN
4,WM,Santuari de Queralt,A,"42,10736","1,8267",POINT (1.8267 42.10736),Santuari de la Mare de DÃ©u de Queralt,80229,1.169,Berga,14,BerguedÃ,8,Barcelona,1,XEMA,2,Operativa,23/01/2002,NaN


In [51]:
df_variables = pd.read_csv(url+"Variables_meteo.csv", sep = ",", header=0, encoding='latin1')
df_variables.head(5)

,CODI_VARIABLE,NOM_VARIABLE,UNITAT,ACRONIM,CODI_TIPUS_VAR,DECIMALS
0,1,PressiÃ³ atmosfÃ¨rica mÃ xima,hPa,Px,DAT,1
1,2,PressiÃ³ atmosfÃ¨rica mÃ­nima,hPa,Pn,DAT,1
2,3,Humitat relativa mÃ xima,%,HRx,DAT,0
3,30,Velocitat del vent a 10 m (esc.),m/s,VV10,DAT,1
4,31,DirecciÃ³ de vent 10 m (m. 1),Â°,DV10,DAT,0


In [ ]:
df = pd.read_csv(url+"Meteo_detall_Vic_XO.csv", sep = ",", header=0, encoding='latin1')
df.sort_values(by='DATA_LECTURA').head(5)

##🟢 **Exploració inicial del dataset**

### **1. Obtenir les dimensions del dataframe**

In [35]:

print ("Número de files: ", df.shape[0])
print ("Número de columnes: ", df.shape[1])

Número de files:  227278
Número de columnes:  8


### **2. Tipus de dades**

In [36]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 227278 entries, 0 to 227277
Data columns (total 8 columns):
 #   Column         Non-Null Count   Dtype 
---  ------         --------------   ----- 
 0   ID             227278 non-null  object
 1   CODI_ESTACIO   227278 non-null  object
 2   CODI_VARIABLE  227278 non-null  int64 
 3   DATA_LECTURA   227278 non-null  object
 4   DATA_EXTREM    104892 non-null  object
 5   VALOR_LECTURA  227278 non-null  object
 6   CODI_ESTAT     104934 non-null  object
 7   CODI_BASE      227278 non-null  object
dtypes: int64(1), object(7)
memory usage: 13.9+ MB


### **3. Detecció de duplicats**

In [37]:
#Obtenim una llista de valors booleans on les files duplicades contenen valor True(1) i les altres False (0).
files_duplicades = df.duplicated(keep = False)
# Sumem els valors de la llista obtinguda.
# Com que les files duplicades tenen valor True (1) obtenim el número total de files duplicades.
print("Número de files duplicades: ", files_duplicades.sum())
# Mostrem les files duplicades ordenades per DNI.
mostrar_dataframe(df[files_duplicades].sort_values(by='ID'), "FILES DUPLICADES")

Número de files duplicades:  0

FILES DUPLICADES


,ID,CODI_ESTACIO,CODI_VARIABLE,DATA_LECTURA,DATA_EXTREM,VALOR_LECTURA,CODI_ESTAT,CODI_BASE


In [38]:
# Obtenim una llista de valors booleans on les files amb clau duplicada contenen valor True(1) i les altres False (0).
clau_duplicada = df.duplicated(subset=['CODI_ESTACIO', 'CODI_VARIABLE','DATA_LECTURA'], keep= False)
# Sumem els valors de la llista per obtenir el número total de registres amb DNI duplicat.
print("Número de claus duplicades: ", clau_duplicada.sum())
# Mostrem els registres amb clau duplicada ordenats per CODI EOI
mostrar_dataframe(df[clau_duplicada].sort_values(by='ID'), "CLAUS DUPLICADES")

Número de claus duplicades:  0

CLAUS DUPLICADES


,ID,CODI_ESTACIO,CODI_VARIABLE,DATA_LECTURA,DATA_EXTREM,VALOR_LECTURA,CODI_ESTAT,CODI_BASE


### **4. Valors Nuls**

In [39]:
print("Valors nuls" )
print(df.isna().sum())

Valors nuls
ID                    0
CODI_ESTACIO          0
CODI_VARIABLE         0
DATA_LECTURA          0
DATA_EXTREM      122386
VALOR_LECTURA         0
CODI_ESTAT       122344
CODI_BASE             0
dtype: int64


##🟢 **Transform: transforma les dades brutes (raw data) per obtenir un conjunt de dades consistent.**

### **Correcció de problemes de codificació i accentuació**

In [52]:
def descodificar (text):
# Funció que retorna el text que rep com a paràmetre sense errors.

  if isinstance (text, str): # Si el text és un string

    # Pas 1: Intentar 'desescapar' les seqüències Unicode literals (per exemple, '\u00f1' -> 'ñ')
    # Utilitzem r'\u' per assegurar que Python interpreta '\u' literalment i no com una seqüència d'escapament Unicode incompleta.
    if r'\u' in text:
        try:
            # codecs.decode converteix les seqüències literals \uXXXX al seu caràcter Unicode
            text = codecs.decode(text, 'unicode_escape')
        except (UnicodeDecodeError):
            pass # Si falla, no és una seqüència d'escapament literal o està mal formatada, continuem.

    # Pas 2: Intentar corregir el 'mojibake' d'una mala descodificació de latin1 a utf-8 (per exemple, 'Ã±' -> 'ñ')
    try:
      return text.encode('latin-1').decode('utf-8').strip()
    except (UnicodeEncodeError, UnicodeDecodeError):
      # Si l'encoding a latin-1 falla o el decoding des de utf-8 falla,
      # pot ser que ja estigui correcte o sigui irreparable per aquest mètode.
      return text.strip()

  # Si el contingut no és un string, no intenta descodificar i retorna el contingut original
  return text

In [53]:
def correccions_codificacio (df, titol):

  print(titol)
  # Correcció dels noms de les columnes
  print ("    Corregim els noms de les columnes...")
  df.columns = [descodificar(col) for col in df.columns]

  # Correcció dels contingut de les columnes
  print ("    Corregim els valors de text de les columnes 'object'...")
  for col in df.columns:
    if df[col].dtype == 'object':
      df[col] = df[col].apply(descodificar)

In [55]:
  correccions_codificacio (df_estacions, "CORRECIÓ PROBLEMES DE CODIFICACIÓ I ACCENTUACIÓ")
  df_estacions.head(5)

CORRECIÓ PROBLEMES DE CODIFICACIÓ I ACCENTUACIÓ
    Corregim els noms de les columnes...
    Corregim els valors de text de les columnes 'object'...


,CODI_ESTACIO,NOM_ESTACIO,CODI_TIPUS,LATITUD,LONGITUD,Georeferència,EMPLACAMENT,CODI_MUNICIPI,ALTITUD,NOM_MUNICIPI,CODI_COMARCA,NOM_COMARCA,CODI_PROVINCIA,NOM_PROVINCIA,CODI_XARXA,NOM_XARXA,CODI_ESTAT,NOM_ESTAT,DATA_ALTA,DATA_BAIXA
0,X2,Barcelona - Zoo,A,"41,38943","2,18847",POINT (2.18847 41.38943),Zoo,80193,7,Barcelona,13,Barcelonès,8,Barcelona,1,XEMA,1,Desmantellada,22/09/2006,03/10/2024
1,UH,el Montmell,A,"41,34172","1,48769",POINT (1.48769 41.34172),Finca Vallflor - Aigua Viva,430901,547,El Montmell,12,Baix Penedès,43,Tarragona,1,XEMA,2,Operativa,13/07/1995,NaN
2,VI,Lleida,A,"41,62918","0,59622",POINT (0.59622 41.62918),Av. Alcalde Rovira Roure,251207,190,Lleida,33,Segrià,25,Lleida,1,XEMA,1,Desmantellada,30/03/1990,15/06/2000
3,WT,Malgrat de Mar,A,"41,64706","2,75659",POINT (2.75659 41.64706),Camí de la Pomereda,81108,3,Malgrat de Mar,21,Maresme,8,Barcelona,1,XEMA,2,Operativa,04/05/2005,NaN
4,WM,Santuari de Queralt,A,"42,10736","1,8267",POINT (1.8267 42.10736),Santuari de la Mare de Déu de Queralt,80229,1.169,Berga,14,Berguedà,8,Barcelona,1,XEMA,2,Operativa,23/01/2002,NaN


In [56]:
  correccions_codificacio (df_variables, "CORRECIÓ PROBLEMES DE CODIFICACIÓ I ACCENTUACIÓ")
  df_variables.head(5)

CORRECIÓ PROBLEMES DE CODIFICACIÓ I ACCENTUACIÓ
    Corregim els noms de les columnes...
    Corregim els valors de text de les columnes 'object'...


,CODI_VARIABLE,NOM_VARIABLE,UNITAT,ACRONIM,CODI_TIPUS_VAR,DECIMALS
0,1,Pressió atmosfèrica màxima,hPa,Px,DAT,1
1,2,Pressió atmosfèrica mínima,hPa,Pn,DAT,1
2,3,Humitat relativa màxima,%,HRx,DAT,0
3,30,Velocitat del vent a 10 m (esc.),m/s,VV10,DAT,1
4,31,Direcció de vent 10 m (m. 1),°,DV10,DAT,0


### **Convertim data a format datetime**

In [57]:
# Convertir les columnes de data a tipus datetime real de Pandas
# 'errors="coerce"' és molt útil perquè si hi ha cel·les buides (NaN) o textos estranys,
# les converteixi automàticament a NaT (Not a Time) sense penjar el programa.


# 1. Specifiquem el format exacte del Meteocat: Dia/Mes/Any Hora(12h):Minuts:Segons AM/PM
format_meteocat = "%d/%m/%Y %I:%M:%S %p"

df["DATA_LECTURA"] = pd.to_datetime(
    df["DATA_LECTURA"], format=format_meteocat, errors="coerce"
)
df["DATA_EXTREM"] = pd.to_datetime(
    df["DATA_EXTREM"], format=format_meteocat, errors="coerce"
)


### **Convertim valor lectura a float**

In [58]:
# 2. Convertir VALOR_LECTURA a float (número decimal)
# Primer passem a string per si de cas, substituïm la coma ',' per un punt '.' i convertim a float
df["VALOR_LECTURA"] = (
    df["VALOR_LECTURA"]
    .astype(str)
    .str.replace(",", ".", regex=False)
    .astype(float)
)


### **Comprovem tipus de dades**

In [59]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 227278 entries, 0 to 227277
Data columns (total 8 columns):
 #   Column         Non-Null Count   Dtype         
---  ------         --------------   -----         
 0   ID             227278 non-null  object        
 1   CODI_ESTACIO   227278 non-null  object        
 2   CODI_VARIABLE  227278 non-null  int64         
 3   DATA_LECTURA   227278 non-null  datetime64[ns]
 4   DATA_EXTREM    104892 non-null  datetime64[ns]
 5   VALOR_LECTURA  227278 non-null  float64       
 6   CODI_ESTAT     104934 non-null  object        
 7   CODI_BASE      227278 non-null  object        
dtypes: datetime64[ns](2), float64(1), int64(1), object(4)
memory usage: 13.9+ MB


In [60]:
df.sort_values(by='DATA_LECTURA').head(5)

,ID,CODI_ESTACIO,CODI_VARIABLE,DATA_LECTURA,DATA_EXTREM,VALOR_LECTURA,CODI_ESTAT,CODI_BASE
9787,XO330101250000,XO,33,2025-01-01,NaT,96.0,NaN,SH
9795,XO720101250000,XO,72,2025-01-01,2025-01-01 00:00:00,0.0,NaN,SH
9794,XO510101250000,XO,51,2025-01-01,NaT,193.0,NaN,SH
9793,XO500101250000,XO,50,2025-01-01,2025-01-01 00:29:00,1.5,NaN,SH
9792,XO440101250000,XO,44,2025-01-01,2025-01-01 00:27:00,95.0,NaN,SH


# **💢 EDA (Exploratory Data Analysis)**

### **Dades de les estacions**

In [61]:
df_estacions[df_estacions['CODI_ESTACIO']=='XO']

,CODI_ESTACIO,NOM_ESTACIO,CODI_TIPUS,LATITUD,LONGITUD,Georeferència,EMPLACAMENT,CODI_MUNICIPI,ALTITUD,NOM_MUNICIPI,CODI_COMARCA,NOM_COMARCA,CODI_PROVINCIA,NOM_PROVINCIA,CODI_XARXA,NOM_XARXA,CODI_ESTAT,NOM_ESTAT,DATA_ALTA,DATA_BAIXA
240,XO,Vic,A,"41,93498","2,23984",POINT (2.23984 41.93498),Vic,82981,500,Vic,24,Osona,8,Barcelona,1,XEMA,2,Operativa,22/12/2011,NaN


### **Tipus de variables meteorològiques mesurades**

In [68]:
df_variables.head(30)

,CODI_VARIABLE,NOM_VARIABLE,UNITAT,ACRONIM,CODI_TIPUS_VAR,DECIMALS
0,1,Pressió atmosfèrica màxima,hPa,Px,DAT,1
1,2,Pressió atmosfèrica mínima,hPa,Pn,DAT,1
2,3,Humitat relativa màxima,%,HRx,DAT,0
3,30,Velocitat del vent a 10 m (esc.),m/s,VV10,DAT,1
4,31,Direcció de vent 10 m (m. 1),°,DV10,DAT,0
5,32,Temperatura,°C,T,DAT,1
6,33,Humitat relativa,%,HR,DAT,0
7,34,Pressió atmosfèrica,hPa,P,DAT,1
8,35,Precipitació,mm,PPT,DAT,1
9,36,Irradiància solar global,W/m²,RS,DAT,0


### **Anàlisi per estació i variable**

**Estacions:**
*   XO: Vic

**Variables:**
*   30: Velocitat del vent a 10 m
*   31: Direcció de vent 10 m (m.1)
*   32: Temperatura
*   33: Humitat relativa
*   34: Pressió atmosfèrica
*   35: Precipitació
*   33: Irradiància solar global

In [ ]:
CodiEstacio = 'XO'
Variable = '32'

### **Número de registres per estació, variable i any**

In [66]:
df.groupby(['CODI_ESTACIO', 'CODI_VARIABLE',  pd.to_datetime(df['DATA_LECTURA'], format='%d/%m/%Y').dt.year ])['ID'].count()

CODI_ESTACIO  CODI_VARIABLE  DATA_LECTURA
XO            3              2025            17485
              30             2025            17479
              31             2025            17476
              32             2025            17485
              33             2025            17485
              35             2025            17486
              36             2025            17485
              40             2025            17485
              42             2025            17485
              44             2025            17485
              50             2025            17478
              51             2025            17478
              72             2025            17486
Name: ID, dtype: int64

### **Estadístiques per estació i variable**

In [73]:
df[df['CODI_VARIABLE'] == 32]['VALOR_LECTURA'].describe()

,VALOR_LECTURA
count,17485.000000
mean,13.809534
std,8.306527
min,-7.200000
25%,7.600000
50%,13.000000
75%,19.500000
max,37.800000
